# 그림 스토리 단어 미션 — 이야기 AI 파인튜닝 (Gemma 4 E2B)

이 노트북은 모범 이야기(이 저장소의 `tools/finetune/train.jsonl`, 일반 사전 낱말로 Claude가 쓴 것)로 Gemma 4 E2B를 조금 더 가르치고,
앱이 크롬에서 쓰는 꼴(MLC q4f16_1)로 바꿔 **내 허깅페이스 계정에 공개로** 올려요.

**할 일 (처음 한 번)**
1. 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 고르고 저장.
2. 왼쪽 **열쇠 모양(보안 비밀)** → **새 보안 비밀 추가**: 이름 `HF_TOKEN`, 값은 허깅페이스 **쓰기(write)** 토큰, **노트북 액세스** 켜기.
   (토큰은 여기에만 넣어요. 채팅이나 다른 곳에 적지 않아요.)
3. **런타임 → 모두 실행**. 40~70분쯤 걸려요. 창을 닫지 말고 두세요.
4. 맨 아래에 나오는 **모델 주소**(https://huggingface.co/…)를 Claude에게 알려 주세요.

중간에 빨간 오류로 멈추면 그 셀의 마지막 몇 줄을 Claude에게 보여 주세요.

## 1. 그래픽 칩 확인과 설정

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU가 없어요: 런타임 → 런타임 유형 변경 → T4 GPU로 바꾸고 다시 '모두 실행' 해 주세요"
print("GPU:", torch.cuda.get_device_name(0))

BRANCH = "claude/word-photo-game-app-16smgx"          # 모범 이야기와 변환 도구가 있는 브랜치
REPO_URL = "https://github.com/areweloststars/-.git"
BASE = "google/gemma-4-E2B-it"                        # 원본 (Apache 2.0)
BASE_REV = "3e22461f65e89153144f8adb70e3b8c2cc9845a7"  # 앱의 크롬용 모델을 만든 판
MLC_BASE = "mlc-ai/gemma-4-E2B-it-q4f16_1-MLC"        # 앱이 지금 쓰는 크롬용 모델
OUT_NAME = "gemma-4-E2B-it-wordstory-q4f16_1-MLC"     # 내 허깅페이스에 만들 저장소 이름
EPOCHS = 3                                            # 모범 이야기를 몇 번 되풀이해 배울지
LORA_R = 16

## 2. 학습 도구 설치 (Unsloth 공식 Gemma 4 노트북과 같은 설치)

In [ ]:
%%capture
import os
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
import importlib.metadata as _md
try: _tv = tuple(int(x) for x in _md.version("torch").split("+")[0].split(".")[:2])
except Exception: _tv = (2, 11)
_torchcodec = {(2, 11): "torchcodec>=0.11.0", (2, 10): "torchcodec==0.10.*", (2, 9): "torchcodec==0.9.*", (2, 8): "torchcodec==0.7.*", (2, 7): "torchcodec==0.5.*"}.get(_tv, "torchcodec>=0.12.0" if _tv >= (2, 12) else "torchcodec")
!pip install "{_torchcodec}"
!pip install --no-deps --upgrade timm
import torch; torch._dynamo.config.recompile_limit = 64;

## 3. 허깅페이스 토큰 확인, 모범 이야기와 크롬용 모델 받기
학습 전에 확인할 것을 먼저 확인해요 (여기서 멈추면 오래 기다리지 않아도 돼요).

In [ ]:
import json, subprocess, sys
from google.colab import userdata
from huggingface_hub import HfApi, snapshot_download

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise SystemExit("보안 비밀 HF_TOKEN을 못 읽었어요. 왼쪽 열쇠 모양에서 이름이 HF_TOKEN인지, '노트북 액세스'가 켜졌는지 보세요. (" + type(e).__name__ + ")")
api = HfApi(token=HF_TOKEN)
me = api.whoami()
role = (me.get("auth") or {}).get("accessToken", {}).get("role")
assert role != "read", "읽기 토큰이에요. 허깅페이스에서 쓰기(write) 토큰을 새로 만들어 HF_TOKEN에 넣어 주세요"
REPO_ID = f"{me['name']}/{OUT_NAME}"
print("올릴 곳:", "https://huggingface.co/" + REPO_ID)

def run(*cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

run("rm", "-rf", "repo")
run("git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, "repo")
rows = [json.loads(l) for l in open("repo/tools/finetune/train.jsonl", encoding="utf-8") if l.strip()]
tests = json.load(open("repo/tools/finetune/test-prompts.json", encoding="utf-8"))
print("모범 이야기", len(rows), "편 · 재 볼 요청", len(tests), "개")

# 앱이 쓰는 크롬용 모델(약 3GB)을 받고, 변환 도구가 원본과 바이트까지 같게 만드는지 0층으로 확인해요
snapshot_download(MLC_BASE, local_dir="mlc_base")
base_url = f"https://huggingface.co/{BASE}/resolve/{BASE_REV}/model.safetensors"
run(sys.executable, "repo/tools/finetune/mlc_patch.py", "--hf", base_url, "--mlc", "mlc_base", "--verify", "--only", "layers.0.")

## 4. 원본 모델 받기 (약 10GB, 몇 분)

In [ ]:
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name = BASE,
    dtype = None,
    max_seq_length = 1024,
    load_in_4bit = False,
    full_finetuning = False,
    token = HF_TOKEN,
)
tok = getattr(tokenizer, "tokenizer", tokenizer)  # 글자 토크나이저
BOS = tok.bos_token_id
TURN_END = tok.convert_tokens_to_ids("<turn|>")
print("bos", BOS, "· <turn|>", TURN_END)

## 5. 학습 준비
앱의 크롬 모델(WebLLM `gemma4_instruction`)과 똑같은 대화 꼴로 쓰고, 답(이야기) 부분만 배우게 해요.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = LORA_R,
    lora_alpha = LORA_R,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

def chat(user, answer=None):
    # WebLLM gemma4_instruction과 Gemma 4 공식 대화 틀이 만드는 글과 같아요
    s = "<bos><|turn>user\n" + user + "<turn|>\n<|turn>model\n"
    return s if answer is None else s + answer + "<turn|>\n"

texts = [chat(r["messages"][0]["content"], r["messages"][1]["content"]) for r in rows]

def make_trainer(texts):
    t = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = Dataset.from_dict({"text": texts}),
        eval_dataset = None,
        args = SFTConfig(
            dataset_text_field = "text",
            per_device_train_batch_size = 1,
            gradient_accumulation_steps = 4,
            warmup_steps = 5,
            num_train_epochs = EPOCHS,
            learning_rate = 2e-4,
            logging_steps = 5,
            optim = "adamw_8bit",
            weight_decay = 0.001,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = "outputs",
            report_to = "none",
        ),
    )
    return train_on_responses_only(t, instruction_part = "<|turn>user\n", response_part = "<|turn>model\n")

trainer = make_trainer(texts)
ids = list(trainer.train_dataset[0]["input_ids"])
if ids[:2] == [BOS, BOS]:  # 토크나이저가 <bos>를 또 붙였으면 글에서 빼요
    trainer = make_trainer([t[len("<bos>"):] for t in texts])
    ids = list(trainer.train_dataset[0]["input_ids"])
assert ids[0] == BOS and ids.count(BOS) == 1, "<bos>가 맨 앞에 하나여야 해요"
assert TURN_END in ids, "<turn|>이 한 토큰이 아니에요"
labels = trainer.train_dataset[0]["labels"]
learned = tok.decode([i for i, l in zip(ids, labels) if l != -100])
print("배울 부분 (첫 이야기):\n" + learned)
assert learned.strip() and "꼭 넣을 말" not in learned, "답 부분만 배우게 하지 못했어요"

## 6. 학습 (T4에서 10~20분쯤)

In [ ]:
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f}분 · 마지막 loss {stats.training_loss:.3f}")

## 7. 대충 재 보기
시험 단어(묶음 A·B·C × 단어 1~5개)로 이야기를 써 봐요. 뜻 찾기는 대충이에요. **진짜 재기는 PC 크롬에서 앱의 점검으로** 해요.

In [ ]:
def write(prompt):
    x = tok(chat(prompt), add_special_tokens = False, return_tensors = "pt").to("cuda")
    out = model.generate(**x, max_new_tokens = 600, do_sample = True, temperature = 0.7, top_p = 0.9,
                         eos_token_id = [tok.eos_token_id, TURN_END])
    return tok.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens = True).strip()

def stem(m):  # 대충: 빌리다 → 빌, 발견하다 → 발견, 용감한 → 용감, 임시의 → 임시
    if m.endswith("하다"): return m[:-2]
    if m.endswith("다"): return m[:-2] if len(m) > 2 else m[:-1]
    if m.endswith(("한", "의")): return m[:-1]
    return m

good = 0
for t in tests:
    story = write(t["prompt"])
    lines = [l.strip() for l in story.split("\n") if l.strip()]
    body = lines[1:]
    ok = len(body) == t["n"] + 2 and all(stem(m) in body[k + 1] for k, m in enumerate(t["words"]))
    good += ok
    print(f"\n[{t['set']} · 단어 {t['n']}개] {'○' if ok else '✕'} 넣을 말: {', '.join(t['words'])}\n" + story)
print(f"\n대충 재기: {good}/{len(tests)}편 (학습 전 Gemma는 8편에 1편쯤)")

## 8. 합친 모델 저장 (16비트, 약 10GB)

In [ ]:
model.save_pretrained("lora")          # 작은 LoRA 조각 (나중에 다시 바꿀 때 쓰려고 함께 올려요)
tokenizer.save_pretrained("lora")
model.save_pretrained_merged("merged", tokenizer, save_method = "merged_16bit")

## 9. 크롬용으로 바꾸기
앱이 쓰는 크롬용 모델 파일에서 글 층의 가중치만 새로 눌러 담아요 (`tools/finetune/mlc_patch.py`). 모델 라이브러리(.wasm)는 앱 것 그대로 써요.

In [ ]:
import numpy as np, os, shutil
run("rm", "-rf", "mlc_out")
run(sys.executable, "repo/tools/finetune/mlc_patch.py", "--hf", "merged", "--mlc", "mlc_base", "--out", "mlc_out")

# 확인: 조금만 바뀌었어야 해요 (모두 같으면 학습이 안 들어간 것, 많이 다르면 이름·꼴이 어긋난 것)
tc = {t["name"]: (sh["dataPath"], t) for sh in json.load(open("mlc_base/tensor-cache.json"))["records"] for t in sh["records"]}
def nibbles(folder, name):
    path, t = tc[name]
    with open(os.path.join(folder, path), "rb") as f:
        f.seek(t["byteOffset"]); a = np.frombuffer(f.read(t["nbytes"]), dtype=np.uint32)
    return np.stack([(a >> (4 * j)) & 15 for j in range(8)])
same = [float((nibbles("mlc_base", n) == nibbles("mlc_out", n)).mean())
        for n in ["language_model.layers.0.self_attn.q_proj.q_weight", "language_model.layers.10.mlp.down_proj.q_weight", "language_model.layers.30.mlp.gate_up_proj.q_weight"]]
print("원본과 같은 4비트 값 비율:", [f"{s:.3f}" for s in same])
assert all(0.5 < s < 1.0 for s in same), "바뀐 정도가 이상해요 (1.0이면 학습이 안 들어감, 0.5 밑이면 이름·꼴이 어긋남). 이 줄을 Claude에게 보여 주세요"

## 10. 허깅페이스에 공개로 올리기

In [ ]:
import urllib.request
open("mlc_out/LICENSE", "w").write(urllib.request.urlopen("https://www.apache.org/licenses/LICENSE-2.0.txt").read().decode())
shutil.copytree("lora", "mlc_out/lora-adapter", dirs_exist_ok = True)
open("mlc_out/README.md", "w").write(f"""---
license: apache-2.0
base_model: {BASE}
tags:
- mlc-llm
- web-llm
- gemma4
language:
- ko
---
# {OUT_NAME}

**Modified version** of [{BASE}](https://huggingface.co/{BASE}) (revision `{BASE_REV}`), licensed under Apache 2.0.

- Changes: LoRA fine-tuned (r={LORA_R}, {EPOCHS} epochs, {len(rows)} short Korean children's stories written for this project with common dictionary words)
  so that it writes a short story that uses given words, for the word game in [areweloststars/-](https://github.com/areweloststars/-).
- The merged weights were re-quantized into the file layout of [{MLC_BASE}](https://huggingface.co/{MLC_BASE}) (q4f16_1)
  with `tools/finetune/mlc_patch.py`. Only the text-layer linear weights (q/k/v/o, gate_up, down) differ from that repository;
  use it with the same WebLLM model library (`gemma-4-E2B-it-q4f16_1`).
- `lora-adapter/`: the LoRA adapter before merging.
""")
api.create_repo(REPO_ID, repo_type = "model", private = False, exist_ok = True)
api.upload_folder(folder_path = "mlc_out", repo_id = REPO_ID, commit_message = "Fine-tuned Gemma 4 E2B for word stories (MLC q4f16_1)")
print("\n다 됐어요. 이 주소를 Claude에게 알려 주세요:\n  https://huggingface.co/" + REPO_ID)